# Validate `vibe_coder_final_boss` on a real T4

End-to-end smoke test for the submission. Run on a Colab T4 runtime
(Runtime → Change runtime type → T4 GPU). Total wall time should be
well under the 30-minute eval budget.

What this notebook does:
1. Clones the comma_video_compression_challenge repo + LFS pulls the videos.
2. Installs Python deps via `uv` (CUDA 12.6 group).
3. Clones our submission fork and copies `submissions/vibe_coder_final_boss/` over.
4. Runs `evaluate.sh --device cuda` and prints the report.

If anything fails or the score deviates from `0.22878`, do not submit.

## 1. Sanity check: what GPU did we get?

In [ ]:
!nvidia-smi

## 2. Clone the challenge repo + pull video LFS

git-lfs is preinstalled on Colab. The videos are ~3 GB.

In [ ]:
%cd /content
!git lfs install
!git clone https://github.com/commaai/comma_video_compression_challenge.git
%cd /content/comma_video_compression_challenge
!git lfs pull
!ls -lh videos/ | head -5

## 3. Install Python deps via `uv`

`uv sync --group cu126` matches the official eval runner. ~1-2 minutes.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"
%cd /content/comma_video_compression_challenge
!uv sync --group cu126

## 4. Drop our submission into `submissions/vibe_coder_final_boss/`

Pulled from the BradyMeighan fork. Replace the URL if you push to a different branch.

In [ ]:
%cd /content
!git clone https://github.com/BradyMeighan/comma_video_compression_challenge.git fork
!git -C /content/fork checkout submission/vibe_coder_final_boss
!cp -r /content/fork/submissions/vibe_coder_final_boss /content/comma_video_compression_challenge/submissions/
!ls -la /content/comma_video_compression_challenge/submissions/vibe_coder_final_boss/

## 5. Download the archive (if not already in the fork checkout)

Skip this cell if `archive.zip` is already next to `inflate.sh`. If you uploaded
the archive to a GitHub Release, set the URL below.

In [ ]:
import os, pathlib
ARCHIVE_URL = "https://github.com/BradyMeighan/comma_video_compression_challenge/releases/download/vcfb-archive/archive.zip"
DEST = "/content/comma_video_compression_challenge/submissions/vibe_coder_final_boss/archive.zip"
if not os.path.exists(DEST) or os.path.getsize(DEST) < 1000:
    !curl -L -o {DEST} {ARCHIVE_URL}
print("size:", os.path.getsize(DEST), "bytes (expected ~197,160)")

## 6. Run the official `evaluate.sh` against our submission

This calls `inflate.sh` which compiles the C++ codec, decodes mask + pose +
model, runs the generator, applies the sidecar, and writes 0.raw. Then
evaluate.py runs the SegNet/PoseNet eval and writes report.txt.

On a T4 this should take 3-5 minutes total.

In [ ]:
%cd /content/comma_video_compression_challenge
!source .venv/bin/activate && time bash evaluate.sh \
    --submission-dir ./submissions/vibe_coder_final_boss \
    --device cuda 2>&1 | tail -40

## 7. Print the report and double-check the score

Expected: `Final score = 0.23` (rounds from 0.22878).

In [ ]:
!cat /content/comma_video_compression_challenge/submissions/vibe_coder_final_boss/report.txt